# 02 EDA – liczebności, braki i długości (surowe dane)

## 1. Konfiguracja

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import (APA_STYLE, CYCLES, EDA_SUBREDDIT_STYLES, PHASES, POLITICAL_SUBREDDITS, PROCESSED_DIR, REFERENCE_SUBREDDITS,
                        SUBREDDITS)
from src.weeks import assign_phase

SCAN_DIR = PROCESSED_DIR / "scan"
EDA_DIR = PROCESSED_DIR / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)
TARGET = 300  # docelowa liczba komentarzy ≥ 5 słów na tydzień
SMOOTH_WEEKS = 5
plt.rcParams.update(APA_STYLE)
plt.rcParams.update({"figure.dpi": 110})


def draw_phases(ax, cycle):
    """Tło panelu: faza szczytowa (żółte pole) i początki pozostałych faz (kropkowane linie)."""
    for phase, start, end in PHASES[cycle]:
        if phase == "szczytowa":
            ax.axvspan(pd.Timestamp(start), pd.Timestamp(end) + pd.Timedelta(days=1), color="#f2e5c6", linewidth=0)
        elif phase != "bazowa":
            ax.axvline(pd.Timestamp(start), color="0.6", linewidth=0.6, linestyle=":")


def plot_weekly(table, column, title, ylabel, log=False, smooth=False, line=None):
    """Wartość tygodniowa w czasie: panel na cykl, linia na subreddit polityczny."""
    fig, axes = plt.subplots(len(CYCLES), 1, figsize=(10, 12), sharey=True)
    for ax, cycle in zip(axes, CYCLES):
        draw_phases(ax, cycle)
        for subreddit in POLITICAL_SUBREDDITS:
            rows = table[(table["subreddit"] == subreddit) & (table["cycle"] == cycle)]
            color, linestyle = EDA_SUBREDDIT_STYLES[subreddit]
            values = rows[column]
            if smooth:
                ax.plot(rows["week_start"], values, color=color, linewidth=0.6, alpha=0.3)
                values = values.rolling(SMOOTH_WEEKS, center=True, min_periods=1).mean()
            ax.plot(rows["week_start"], values, color=color, linestyle=linestyle, linewidth=1.3, label=f"r/{subreddit}")
        if line:
            ax.axhline(line, color="0.3", linestyle="--", linewidth=0.8)
        if log:
            ax.set_yscale("log")
        ax.set_title(f"Cykl {cycle}", loc="left")
    fig.suptitle(title, x=0.01, ha="left", fontsize=12)
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, 0.965), ncol=4)
    fig.supxlabel("Tydzień (żółte tło = faza szczytowa, kropkowane linie = początki faz)")
    fig.supylabel(ylabel)
    plt.tight_layout(rect=(0.02, 0, 1, 0.95))
    plt.show()

## 2. Wczytanie tabel ze skanu

In [2]:
files = pd.read_csv(SCAN_DIR / "files.csv", parse_dates=["first_utc", "last_utc"])
weekly = pd.read_csv(SCAN_DIR / "weekly.csv", parse_dates=["week_start"])
schema = pd.read_csv(SCAN_DIR / "schema.csv")
author_percentiles = pd.read_csv(SCAN_DIR / "author_percentiles.csv")
author_thresholds = pd.read_csv(SCAN_DIR / "author_thresholds.csv")
weekly["phase"] = assign_phase(weekly["week_start"], weekly["cycle"])
weekly["camp"] = weekly["subreddit"].map(SUBREDDITS)
files["file"] = files["subreddit"] + " " + files["cycle"].astype(str)
political = weekly[weekly["subreddit"].isin(POLITICAL_SUBREDDITS)]
print(files.shape, weekly.shape, f"| rekordów łącznie: {files['n_records'].sum():,} | {files['size_mb'].sum() / 1000:.1f} GB")

(30, 13) (2826, 15) | rekordów łącznie: 20,578,055 | 29.0 GB


## 3. Inwentarz plików

In [3]:
files[["subreddit", "cycle", "size_mb", "n_records", "n_bad_lines", "n_outside_window", "n_empty_weeks", "first_utc", "last_utc"]]

,subreddit,cycle,size_mb,n_records,n_bad_lines,n_outside_window,n_empty_weeks,first_utc,last_utc
0,democrats,2012,10.9,15002,0,0,0,2011-03-21 05:24:11,2013-03-16 22:42:39
1,democrats,2016,73.7,109049,0,0,0,2015-03-16 00:00:16,2017-03-18 23:51:11
2,democrats,2020,455.9,351984,0,0,0,2019-03-18 00:14:15,2021-03-20 23:59:09
3,democrats,2024,1999.6,1171188,0,0,0,2023-03-20 00:01:05,2025-03-15 23:59:33
4,Liberal,2012,20.7,25914,0,0,0,2011-03-21 00:20:55,2013-03-17 22:26:04
5,Liberal,2016,44.2,58705,0,0,0,2015-03-16 00:28:42,2017-03-19 23:34:14
6,Liberal,2020,98.8,76403,0,0,0,2019-03-18 00:12:08,2021-03-20 23:29:49
7,Liberal,2024,291.7,156937,0,0,0,2023-03-20 00:26:54,2025-03-16 23:59:10
8,Republican,2012,78.0,101372,0,0,0,2011-03-21 13:06:07,2013-03-16 23:54:25
9,Republican,2016,99.5,143316,0,0,0,2015-03-16 00:16:52,2017-03-18 23:43:56


**Rozmiar plików**

- Razem 30 plików, 20,6 mln rekordów i 29 GB. Najmniejsze są r/democrats 2012 (15 tys.) i r/Liberal 2012 (26 tys.), największe r/Conservative 2020 (6,4 mln) i 2024 (4,4 mln).
- r/Conservative ma 13× więcej rekordów niż r/Liberal w 2012, 15× w 2016, 83× w 2020 i 28× w 2024.

Subreddity są skrajnie nierówne, a obóz republikański to w praktyce r/Conservative. Bez równej próby na tydzień
(limit 300 komentarzy) jeden subreddit zdominowałby wynik swojego obozu.

## 4. Schemat pól

In [5]:
share = schema.pivot_table(index="field", columns=["subreddit", "cycle"], values="share", fill_value=0)
needed = ["id", "author", "body", "created_utc", "retrieved_on", "link_id", "parent_id", "distinguished", "subreddit", "_meta"]
needed = [field for field in needed if field in share.index]
print(f"pól w danych: {len(share)}, używamy: {len(needed)}")
share.loc[needed].min(axis=1).round(3).to_frame("najmniejszy_udzial_w_pliku")

pól w danych: 82, używamy: 10


,najmniejszy_udzial_w_pliku
field,
id,1.000
author,1.000
body,1.000
created_utc,1.000
retrieved_on,0.894
link_id,1.000
parent_id,1.000
distinguished,1.000
subreddit,1.000


**Kompletność pól, których używamy**

- Z 82 pól w danych używamy 10. Osiem z nich jest w 100% rekordów każdego pliku.
- Dwa wyjątki: `retrieved_on` jest puste w 7–11% rekordów z 2020, a `_meta` istnieje tylko w 2024 (92–99% rekordów, w pozostałych latach 0).

`removed_later` da się więc policzyć tylko dla 2024, a w starszych cyklach brak tej flagi nie znaczy, że nic nie usunięto.
Opóźnienie archiwizacji w 2020 liczymy z ok. 90% rekordów.

## 5. Liczebność tygodniowa w czasie

In [ ]:
plot_weekly(weekly.assign(n_ge_5w=weekly["n_ge_5w"].clip(lower=1)), "n_ge_5w",
            "Liczba komentarzy ≥ 5 słów na tydzień przed czyszczeniem",
            f"Komentarze ≥ 5 słów na tydzień (skala log, linia przerywana = {TARGET})", log=True, line=TARGET)

**Jak czytać**
- Jeden panel na cykl, linia na subreddit polityczny (niebieskie: demokraci, czerwone: republikanie). Oś Y w skali logarytmicznej: komentarze od 5 słów na tydzień w surowych danych.
- Linia przerywana to cel próby (300 komentarzy na tydzień). Żółte tło to faza szczytowa, kropkowane linie pionowe to początki pozostałych faz.

**Co widać**
- Wolumen narasta przed wyborami i ma ostry pik w tygodniu wyborów. W 2012 i 2020 r/Conservative rośnie przez cały cykl (z ok. 12 tys. do ponad 70 tys. na tydzień w 2020), a w 2024 jest płaski lub spada.
- r/democrats 2024 skacze w lipcu 2024 z ok. 3 tys. do ok. 20 tys. na tydzień i zostaje wysoko do wyborów.
- r/Republican 2020 ma w danych dziury: pusty tydzień 17.08.2020 oraz głębokie spadki 9.11.2020 i 11.01.2021 (omówione w rozdz. 6).
- Fazy mają różną długość: od 8 tygodni (powyborcza) do ponad 60 (przedwyborcza).
- Faza szczytowa jest zatem fazą największego i zmieniającego się wolumenu. Jej średnia składa się z tygodni o bardzo różnej liczebności i prawdopodobnie z innego składu autorów niż baza.

## 6. Dziury w danych: tygodnie z nagle niższą liczbą rekordów

Tydzień uznaję za dziurę, gdy ma mniej niż 35% mediany z 9 tygodni wokół niego (liczone na wszystkich rekordach).

In [17]:
weekly = weekly.sort_values(["subreddit", "cycle", "week_start"]).reset_index(drop=True)
weekly["neighbour_median"] = weekly.groupby(["subreddit", "cycle"])["n_raw"].transform(lambda values: values.rolling(9, center=True, min_periods=5).median())
weekly["gap_ratio"] = weekly["n_raw"] / weekly["neighbour_median"]
gaps = weekly[weekly["gap_ratio"] < 0.35][["subreddit", "cycle", "week_start", "phase", "n_raw", "neighbour_median", "gap_ratio"]]
gaps.to_csv(EDA_DIR / "raw_gaps.csv", index=False)
gaps.round({"gap_ratio": 2, "neighbour_median": 0})

,subreddit,cycle,week_start,phase,n_raw,neighbour_median,gap_ratio
505,Liberal,2012,2012-11-19,przejsciowa,170,519.0,0.33
836,Republican,2012,2011-03-21,bazowa,17,56.0,0.30
837,Republican,2012,2011-03-28,bazowa,7,62.0,0.11
1119,Republican,2020,2020-08-17,przedwyborcza,0,5912.0,0.00
1120,Republican,2020,2020-08-24,szczytowa,430,5956.0,0.07
1131,Republican,2020,2020-11-09,przejsciowa,572,8237.0,0.07
1140,Republican,2020,2021-01-11,przejsciowa,686,4345.0,0.16
1681,languagelearning,2012,2011-05-23,bazowa,77,239.0,0.32
2734,running,2024,2023-06-12,bazowa,1324,4938.0,0.27


In [ ]:
gap_files = list(gaps[["subreddit", "cycle"]].drop_duplicates().itertuples(index=False))
fig, axes = plt.subplots(1, len(gap_files), figsize=(3.4 * len(gap_files), 3.6))
for ax, (subreddit, cycle) in zip(axes, gap_files):
    rows = weekly[(weekly["subreddit"] == subreddit) & (weekly["cycle"] == cycle)]
    marked = rows[rows["gap_ratio"] < 0.35]
    ax.plot(rows["week_start"], rows["n_raw"].clip(lower=1), color="#333333", linewidth=1)
    ax.scatter(marked["week_start"], marked["n_raw"].clip(lower=1), color="#b2182b", s=22, zorder=3)
    ax.set_yscale("log")
    ax.set_title(f"r/{subreddit} {cycle}", loc="left", fontsize=9)
    ax.tick_params(axis="x", rotation=40, labelsize=7)
axes[0].set_ylabel("Rekordy na tydzień (skala log)")
fig.suptitle("Dziury w danych: pliki, w których tydzień ma mniej niż 35% mediany sąsiednich tygodni (czerwone punkty)", x=0.01, ha="left",
             fontsize=12)
plt.tight_layout()
plt.show()

**Jak czytać**
- Każdy panel to plik, w którym jest tydzień z liczbą rekordów poniżej 35% mediany sąsiednich tygodni (okno 9 tygodni). Czerwone punkty to takie tygodnie, oś Y jest w skali logarytmicznej.

**Co widać**
- Znalazłem 9 tygodni. Najpoważniejsze są w r/Republican 2020: pusty tydzień 17.08.2020 (0 rekordów, faza przedwyborcza), potem 24.08.2020 (430 wobec ok. 6 tys. zwykle) i jest to **pierwszy tydzień fazy szczytowej**, a także 9.11.2020 (572) i 11.01.2021 (686) w fazie przejściowej.
- Pozostałe to małe pliki z 2012: r/Republican (21 i 28.03.2011, początek subreddytu), r/Liberal (19.11.2012, 170 rekordów) i r/languagelearning (23.05.2011).
- r/running 2024: tydzień 12.06.2023 ma 27% normalnej liczby rekordów. To zbiega się z protestem Reddita przeciw zmianom API (12–14.06.2023), ale tej przyczyny nie sprawdzałem.
- Dziury w r/Republican 2020 nie są spadkiem aktywności, tylko brakiem danych w archiwum. Tygodnie z dziurą mają za mało rekordów, żeby próba była reprezentatywna dla tygodnia. Warto je oznaczyć w analizie wrażliwości (np. pominąć w H1–H3).

## 7. Usunięte komentarze

In [19]:
total = weekly.groupby(["subreddit", "cycle"])[["n_raw", "n_deleted", "n_moderator", "n_removed_later", "n_content", "n_link"]].sum()
removed = pd.DataFrame({"share_deleted": total["n_deleted"] / total["n_raw"],
                        "share_moderator": total["n_moderator"] / total["n_raw"],
                        "share_removed_later": total["n_removed_later"] / total["n_raw"],
                        "share_link": total["n_link"] / total["n_content"]})
removed.to_csv(EDA_DIR / "raw_removed.csv")
removed.round(3)

share_deleted  share_moderator  share_removed_later  \
subreddit        cycle                                                        
Conservative     2012           0.086            0.002                0.000   
                 2016           0.113            0.007                0.000   
                 2020           0.251            0.004                0.000   
                 2024           0.509            0.001                0.076   
Liberal          2012           0.071            0.002                0.000   
                 2016           0.085            0.007                0.000   
                 2020           0.233            0.002                0.000   
                 2024           0.236            0.003                0.033   
Republican       2012           0.084            0.001                0.000   
                 2016           0.088            0.010                0.000   
                 2020           0.165            0.074                0.000   
                 2024           0.182            0.069                0.082   
democrats        2012           0.067            0.001                0.000   
                 2016           0.105            0.021                0.000   
                 2020           0.209            0.039                0.000   
                 2024           0.249            0.020                0.044   
languagelearning 2012           0.070            0.000                0.000   
                 2014           0.067            0.001                0.000   
                 2016           0.043            0.012                0.000   
                 2018           0.047            0.014                0.000   
                 2020           0.068            0.026                0.000   
                 2022           0.053            0.028                0.000   
                 2024           0.014            0.044                0.029   
running          2012           0.054            0.000                0.000   
                 2014           0.048            0.001                0.000   
                 2016           0.025            0.006                0.000   
                 2018           0.028            0.010                0.000   
                 2020           0.045            0.011                0.000   
                 2022           0.041            0.024                0.000   
                 2024           0.013            0.138                0.021   

                        share_link  
subreddit        cycle              
Conservative     2012        0.072  
                 2016        0.059  
                 2020        0.039  
                 2024        0.027  
Liberal          2012        0.076  
                 2016        0.070  
                 2020        0.047  
                 2024        0.031  
Republican       2012        0.070  
                 2016        0.070  
                 2020        0.121  
                 2024        0.107  
democrats        2012        0.073  
                 2016        0.093  
                 2020        0.081  
                 2024        0.059  
languagelearning 2012        0.122  
                 2014        0.106  
                 2016        0.097  
                 2018        0.081  
                 2020        0.078  
                 2022        0.064  
                 2024        0.078  
running          2012        0.070  
                 2014        0.067  
                 2016        0.063  
                 2018        0.049  
                 2020        0.042  
                 2022        0.050  
                 2024        0.162

In [20]:
removed_by_phase = weekly.groupby(["subreddit", "cycle", "phase"])[["n_raw", "n_deleted"]].sum()
removed_by_phase["share_deleted"] = removed_by_phase["n_deleted"] / removed_by_phase["n_raw"]
removed_by_phase.loc[POLITICAL_SUBREDDITS].to_csv(EDA_DIR / "raw_removed_by_phase.csv")
removed_by_phase.loc[POLITICAL_SUBREDDITS, "share_deleted"].unstack("phase").round(3)

phase               bazowa  powyborcza  przedwyborcza  przejsciowa  szczytowa
subreddit    cycle                                                           
democrats    2012    0.087       0.070          0.062        0.076      0.067
             2016    0.040       0.174          0.081        0.127      0.167
             2020    0.187       0.209          0.208        0.216      0.216
             2024    0.133       0.296          0.194        0.458      0.144
Liberal      2012    0.055       0.057          0.074        0.069      0.082
             2016    0.055       0.171          0.067        0.113      0.113
             2020    0.230       0.209          0.229        0.236      0.268
             2024    0.190       0.308          0.199        0.278      0.193
Republican   2012    0.115       0.086          0.073        0.091      0.093
             2016    0.117       0.125          0.073        0.096      0.091
             2020    0.184       0.133          0.164        0.169      0.178
             2024    0.214       0.307          0.112        0.198      0.151
Conservative 2012    0.114       0.090          0.066        0.108      0.096
             2016    0.139       0.120          0.122        0.080      0.091
             2020    0.169       0.245          0.183        0.378      0.239
             2024    0.345       0.728          0.447        0.592      0.605

### 7.1 Udział `[deleted]` / `[removed]` w cyklach

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, group, title in zip(axes, (POLITICAL_SUBREDDITS, REFERENCE_SUBREDDITS), ("Subreddity polityczne", "Subreddity odniesienia")):
    for subreddit in group:
        rows = removed.loc[subreddit]
        color, linestyle = EDA_SUBREDDIT_STYLES[subreddit]
        ax.plot(rows.index.astype(str), rows["share_deleted"], color=color, linestyle=linestyle, marker="o", label=f"r/{subreddit}")
    ax.set_title(title, loc="left")
    ax.set_xlabel("Cykl / rok")
    ax.legend()
axes[0].set_ylabel("Udział usuniętych komentarzy")
fig.suptitle("Udział usuniętych komentarzy ([deleted] / [removed]) wśród wszystkich rekordów pliku", x=0.01, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

**Jak czytać**
- Oś X: cykl (lub rok referencyjny), oś Y: udział usuniętych komentarzy ([deleted] / [removed]) wśród wszystkich rekordów pliku. Lewy panel: subreddity polityczne, prawy: odniesienia (wspólna oś Y).

**Co widać**
- W subreddytach politycznych udział usuniętych rośnie z cyklem: r/Conservative 8,6% → 11,3% → 25,1% → 50,9%, r/democrats 6,7% → 25%, r/Liberal 7% → 24%, r/Republican 8% → 18%.
- W subreddytach odniesienia jest niski (1–7%) i w 2024 spada (r/running 1,3%, r/languagelearning 1,4%).
- Wzrost dotyczy polityki, nie całego Reddita, więc nie jest efektem ogólnego sposobu archiwizacji. To główne źródło selekcji: w 2024 w r/Conservative widzimy tylko połowę komentarzy, które kiedykolwiek napisano. Nie wiemy, czy usunięte różnią się emocjonalnie od pozostałych.

### 7.2 Udział usuniętych w czasie

In [ ]:
political_weekly = political.assign(share_deleted=political["n_deleted"] / political["n_raw"].replace(0, np.nan))
plot_weekly(political_weekly, "share_deleted",
            "Udział usuniętych komentarzy wśród wszystkich rekordów tygodnia (surowe dane, przed czyszczeniem)",
            f"Udział usuniętych (średnia z {SMOOTH_WEEKS} tygodni, cienka linia = pojedyncze tygodnie)", smooth=True)

In [24]:
political_weekly = political_weekly.sort_values(["subreddit", "cycle", "week_start"])
political_weekly["change"] = political_weekly.groupby(["subreddit", "cycle"])["share_deleted"].diff()
jumps = political_weekly[(political_weekly["change"].abs() > 0.15) & (political_weekly["n_raw"] >= 500)]
jumps = jumps[["subreddit", "cycle", "week_start", "phase", "n_raw", "share_deleted", "change"]]
jumps.to_csv(EDA_DIR / "raw_deleted_jumps.csv", index=False)
print(f"skoków o ponad 15 p.p. w tydzień (tygodnie ≥ 500 rekordów): {len(jumps)}")
jumps.round({"share_deleted": 2, "change": 2})

skoków o ponad 15 p.p. w tydzień (tygodnie ≥ 500 rekordów): 23


,subreddit,cycle,week_start,phase,n_raw,share_deleted,change
1548,Conservative,2020,2020-11-02,przejsciowa,367201,0.43,0.19
1550,Conservative,2020,2020-11-16,przejsciowa,181185,0.22,-0.18
1556,Conservative,2020,2020-12-28,przejsciowa,70812,0.46,0.16
1557,Conservative,2020,2021-01-04,przejsciowa,139821,0.61,0.16
1558,Conservative,2020,2021-01-11,przejsciowa,106950,0.44,-0.17
1559,Conservative,2020,2021-01-18,powyborcza,192241,0.23,-0.21
1580,Conservative,2024,2023-06-12,bazowa,70640,0.24,-0.15
1587,Conservative,2024,2023-07-31,bazowa,28487,0.51,0.24
1609,Conservative,2024,2024-01-01,przedwyborcza,29852,0.48,-0.21
1633,Conservative,2024,2024-06-17,przedwyborcza,25389,0.40,-0.17


**Jak czytać**
- Jeden panel na cykl, linia na subreddit. Cienka blada linia to pojedyncze tygodnie, gruba to średnia z 5 tygodni. Wartość to udział usuniętych komentarzy wśród wszystkich rekordów tygodnia (surowe dane).
- Żółte tło to faza szczytowa, kropkowane linie pionowe to początki pozostałych faz.

**Co widać**
- W 2012 i 2016 udział jest prawie płaski, a w 2020 i 2024 ma **skokowe zmiany** w poszczególnych tygodniach, nie powolny trend.
- r/democrats 2024: z ok. 15% do 45% dokładnie w tygodniu wyborów (4.11.2024), potem w fazach przejściowej i powyborczej 25–50%. r/Conservative 2020: o 19 p.p. w tygodniu wyborów (2.11.2020, do 43%) i do 61% w tygodniu 4.01.2021. r/Liberal 2024: 40–55% w styczniu i w lipcu–sierpniu 2024, a potem z powrotem ok. 20%.
- Pojedyncze tygodnie z 2012 są bardzo zmienne, bo opierają się na kilkudziesięciu rekordach.
- Skoki w ciągu jednego tygodnia wskazują na zdarzenia moderacyjne lub masowe czyszczenie, a nie na naturalne „starzenie się” komentarzy. Przyczyny nie znam. Wiem tylko, że wypadają w tygodniach ważnych dla analizy.

## 8. Autorzy

In [31]:
author_percentiles.round(1)

,subreddit,cycle,n_authors,n_comments,p50,p90,p99,p999,max
0,democrats,2012,2711,13993,1.0,9.0,49.9,161.6,255
1,democrats,2016,8984,97597,2.0,13.0,124.0,852.3,7312
2,democrats,2020,33106,278367,2.0,13.0,94.0,522.6,12679
3,democrats,2024,89892,879196,2.0,17.0,141.0,529.5,22436
4,Liberal,2012,4264,24085,2.0,10.0,58.4,150.4,219
5,Liberal,2016,7722,53710,2.0,12.0,90.8,359.0,659
6,Liberal,2020,10720,58612,2.0,10.0,57.0,177.4,1138
7,Liberal,2024,14909,119866,2.0,18.0,86.0,254.5,1160
8,Republican,2012,8618,92852,2.0,16.0,135.0,511.2,2368
9,Republican,2016,10369,130687,2.0,18.0,145.3,899.9,10623


In [32]:
author_thresholds.pivot_table(index=["subreddit", "cycle"], columns="min_comments", values="share_comments").round(3)

min_comments             50     100    200    300    500    1000   2000   5000
subreddit        cycle                                                        
Conservative     2012   0.594  0.497  0.390  0.339  0.256  0.157  0.082  0.054
                 2016   0.697  0.598  0.498  0.430  0.341  0.248  0.151  0.044
                 2020   0.641  0.545  0.441  0.378  0.304  0.196  0.113  0.035
                 2024   0.670  0.564  0.443  0.368  0.269  0.152  0.065  0.008
Liberal          2012   0.201  0.091  0.026  0.000  0.000  0.000  0.000  0.000
                 2016   0.380  0.259  0.144  0.092  0.023  0.000  0.000  0.000
                 2020   0.270  0.165  0.090  0.078  0.059  0.019  0.000  0.000
                 2024   0.337  0.186  0.097  0.065  0.047  0.019  0.000  0.000
Republican       2012   0.481  0.353  0.226  0.175  0.116  0.062  0.047  0.000
                 2016   0.602  0.491  0.388  0.330  0.274  0.201  0.149  0.132
                 2020   0.356  0.262  0.188  0.161  0.135  0.114  0.100  0.086
                 2024   0.543  0.407  0.296  0.232  0.183  0.165  0.140  0.120
democrats        2012   0.199  0.122  0.018  0.000  0.000  0.000  0.000  0.000
                 2016   0.629  0.532  0.452  0.397  0.337  0.246  0.209  0.141
                 2020   0.480  0.378  0.292  0.245  0.199  0.129  0.103  0.046
                 2024   0.552  0.403  0.266  0.192  0.120  0.063  0.031  0.026
languagelearning 2012   0.291  0.198  0.140  0.086  0.033  0.000  0.000  0.000
                 2014   0.340  0.222  0.121  0.052  0.019  0.019  0.000  0.000
                 2016   0.479  0.347  0.236  0.170  0.115  0.044  0.000  0.000
                 2018   0.401  0.296  0.215  0.165  0.118  0.049  0.000  0.000
                 2020   0.418  0.320  0.230  0.184  0.130  0.084  0.051  0.013
                 2022   0.456  0.328  0.221  0.168  0.106  0.055  0.030  0.017
                 2024   0.457  0.343  0.245  0.196  0.149  0.106  0.070  0.041
running          2012   0.472  0.353  0.237  0.185  0.135  0.064  0.035  0.015
                 2014   0.409  0.297  0.194  0.155  0.102  0.051  0.010  0.000
                 2016   0.653  0.560  0.467  0.414  0.347  0.261  0.194  0.080
                 2018   0.504  0.399  0.303  0.257  0.185  0.108  0.071  0.000
                 2020   0.449  0.339  0.249  0.208  0.161  0.108  0.069  0.026
                 2022   0.375  0.264  0.187  0.150  0.112  0.078  0.034  0.024
                 2024   0.459  0.386  0.322  0.297  0.264  0.232  0.210  0.161

### 8.1 Koncentracja aktywności

In [ ]:
curves = author_thresholds[author_thresholds["subreddit"].isin(POLITICAL_SUBREDDITS)]
fig, axes = plt.subplots(1, len(CYCLES), figsize=(14, 4), sharey=True)
for ax, cycle in zip(axes, CYCLES):
    for subreddit in POLITICAL_SUBREDDITS:
        rows = curves[(curves["subreddit"] == subreddit) & (curves["cycle"] == cycle)]
        color, linestyle = EDA_SUBREDDIT_STYLES[subreddit]
        ax.plot(rows["min_comments"], rows["share_comments"], color=color, linestyle=linestyle, marker="o", markersize=3,
                label=f"r/{subreddit}")
    ax.set_xscale("log")
    ax.set_title(f"Cykl {cycle}", loc="left")
    ax.set_xlabel("Minimalna liczba komentarzy konta")
axes[0].set_ylabel("Udział komentarzy tych kont")
axes[0].legend(fontsize=8)
fig.suptitle("Koncentracja autorów: jaką część komentarzy piszą konta z co najmniej N komentarzami w pliku", x=0.01, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

**Jak czytać**
- Jeden panel na cykl. Oś X (skala logarytmiczna): minimalna liczba komentarzy konta w pliku. Oś Y: jaka część wszystkich komentarzy pochodzi z kont, które mają co najmniej tyle komentarzy. Im wyżej linia, tym bardziej aktywność jest skoncentrowana w kilku kontach.

**Co widać**
- Typowy autor ma 2 komentarze w pliku (mediana; 1 w r/running 2024), ale 99. percentyl to 50–255.
- Konta z ≥ 100 komentarzami piszą **od 9% (r/Liberal 2012) do 60% (r/Conservative 2016)** wszystkich komentarzy. W r/Conservative to 50–60% w każdym cyklu, w r/Liberal 9–26%, w r/democrats 12% (2012) i 38–53% w pozostałych cyklach, w r/Republican 26–49%.
- W 2012 w r/democrats i r/Liberal prawie nie ma kont powyżej 300 komentarzy.
- Rozkład jest skrajnie skośny, a asymetria obozów jest duża: po stronie republikańskiej garstka kont pisze połowę komentarzy, po demokratycznej (r/Liberal) kilkanaście procent. Wybór progu `HIGH_VOLUME_MIN_COMMENTS` zmienia więc skład próby bardziej w obozie republikańskim niż w demokratycznym. Próg warto opisać i sprawdzić analizą wrażliwości.